# Обучение DL-модели

В качетсве основгого фреймворка используем TenserFlow (Keras), для обучения, потому что:
- в библиотеке при обучении сразу считаются ключевые метрики, что упростит логирование
- проще экспериментировать с настройками моделей

## Устанавливаем и импортируем необходимые библиотеки

In [170]:
# !pip install clearml

In [171]:
# !pip install tensorflow

In [172]:
import numpy as np
import pandas as pd

import logging
from clearml import Task


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import mean_absolute_error

## Включаем логирование

По умолчанию будем использовать уровень INFO

In [173]:
logging.basicConfig(
    level = logging.INFO
)

**Подключим логирование в ClearML**

In [174]:
task_1 = Task.init(
    project_name = 'gp5',
    task_name = 'FCN_baseline_v2',
    reuse_last_task_id = False
)

ClearML Task: created new task id=28e0a451daf2422c955781ed0e08ad00


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/2ace20be80174dbc9983cfad0cb686c1/experiments/28e0a451daf2422c955781ed0e08ad00/output/log


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


Зададим конфикурацию для экспериментов с FCN-моделями

In [ ]:
config_to_experiments = {
    'activation': 'relu',
    'dropout': 0.3,
    'hidden_layers': [64],
    'learning_rate': 1e-3,
    'epochs': 50,
    'batch_size': 512,
    'embedding_dim': 32
}

Передаем конфигурацию в ClearML:

In [176]:
task_1.connect(config_to_experiments)

{'activation': 'relu',
 'dropout': 0.3,
 'hidden_layers': [512, 256, 64],
 'learning_rate': 0.001,
 'epochs': 50,
 'batch_size': 512,
 'embedding_dim': 32}

### Загрузка датасетов, полученных после EDA и предобработки

In [177]:
logging.info('....начало загрузки датасетов....')

INFO:root:....начало загрузки датасетов....


In [178]:
df_train = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_train.csv')
df_val = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_val.csv')
df_test = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_test.csv')

In [179]:
logging.info('....загрузка датасетов закончена....')

INFO:root:....загрузка датасетов закончена....


**Определимся с назначением каждого из датасетов**

Предсказываем 'popularity': 0-100

In [180]:
all_columns = df_train.columns.to_list()
all_columns.remove('Unnamed: 0')
all_columns

['popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'artists_amount',
 'new_explicit',
 'track_genre_acoustic',
 'track_genre_afrobeat',
 'track_genre_alt-rock',
 'track_genre_alternative',
 'track_genre_ambient',
 'track_genre_anime',
 'track_genre_black-metal',
 'track_genre_bluegrass',
 'track_genre_blues',
 'track_genre_brazil',
 'track_genre_breakbeat',
 'track_genre_british',
 'track_genre_cantopop',
 'track_genre_chicago-house',
 'track_genre_children',
 'track_genre_chill',
 'track_genre_classical',
 'track_genre_club',
 'track_genre_comedy',
 'track_genre_country',
 'track_genre_dance',
 'track_genre_dancehall',
 'track_genre_death-metal',
 'track_genre_deep-house',
 'track_genre_detroit-techno',
 'track_genre_disco',
 'track_genre_disney',
 'track_genre_drum-and-bass',
 'track_genre_dub',
 'track_genre_dubstep',
 'track

In [181]:
df_train.drop(columns = 'Unnamed: 0', inplace = True)
df_val.drop(columns = 'Unnamed: 0', inplace = True)
df_test.drop(columns = 'Unnamed: 0', inplace = True)

In [182]:
PREDICT_VAR = 'popularity'
all_columns.remove('popularity')

ARTIST_NAME  = 'artist_label'
all_columns.remove('artist_label')

FEATURES_COLUMNS = all_columns

Теперь определение датасетов

In [183]:
X_train = df_train[FEATURES_COLUMNS].astype(float).values
X_val = df_val[FEATURES_COLUMNS].astype(float).values
X_test = df_test[FEATURES_COLUMNS].astype(float).values

In [184]:
Y_train = df_train[PREDICT_VAR].values
Y_val = df_val[PREDICT_VAR].values
Y_test = df_test[PREDICT_VAR].values

Отдельно обрабатываем столбец с именем артиста, поскольку ...

In [185]:
ART_train = df_train[ARTIST_NAME].values
ART_val = df_val[ARTIST_NAME].values
ART_test = df_test[ARTIST_NAME].values

## Строим DL-модель

Используем 2 входа Keras, поскольку в датасете есть столбцы закодированы 2 принципиально разными способома:
- OHE
- LabelEncoding

https://www.kaggle.com/code/hireme/two-inputs-neural-network-using-keras

In [186]:
numbers_input = keras.Input(shape = (X_train.shape[1],), name = 'numbers_feautures_input')

artists_input = keras.Input(shape = (1,), name = 'artists_feauture_input')

amount_of_artist_max = max(
    df_train['artist_label'].max(),
    df_val['artist_label'].max(),
    df_test['artist_label'].max()
) + 1

artist_embedding = layers.Embedding(
    input_dim = amount_of_artist_max,
    output_dim = config_to_experiments['embedding_dim'],
    name = 'artist_embedding'
)(artists_input)

artist_embedding = layers.Flatten()(artist_embedding)


x_full = layers.Concatenate()([numbers_input, artist_embedding])

**Создадим полносвязные слои для нейросети**

In [187]:
for layer in config_to_experiments['hidden_layers']:
    x_full = layers.Dense(
        layer,
        activation = config_to_experiments['activation']
    )(x_full)

    x_full = layers.Dropout(
        config_to_experiments['dropout']
    )(x_full)

output_data = layers.Dense(1, name = 'output_data')(x_full)

**Создадим сам объект модели**

In [188]:
model_task_1 = keras.Model(
    inputs = [numbers_input, artists_input],
    outputs = output_data
)

In [189]:
model_task_1.compile(
    optimizer = keras.optimizers.Adam(learning_rate = config_to_experiments['learning_rate']),
    loss = 'mse',
    metrics = ['mae']
)

Проверим, что все параметры корректны:

In [190]:
model_task_1.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ artists_feauture_i… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ artist_embedding    │ (None, 1, 32)     │    486,912 │ artists_feauture… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numbers_feautures_… │ (None, 130)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 32)        │          0 │ artist_embedding… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 162)       │          0 │ numbers_feauture… │
│ (Concatenate)       │                   │            │ flatten_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 512)       │     83,456 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 512)       │          0 │ dense_18[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 256)       │    131,328 │ dropout_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 256)       │          0 │ dense_19[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 64)        │     16,448 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_20          │ (None, 64)        │          0 │ dense_20[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_data (Dense) │ (None, 1)         │         65 │ dropout_20[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 718,209 (2.74 MB)

 Trainable params: 718,209 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

## Обучение модели

In [191]:
logging.info('Начало обучения модели')
logger = task_1.get_logger()

INFO:root:Начало обучения модели


In [192]:
learning_history = model_task_1.fit(
    [X_train, ART_train], y = Y_train,
    validation_data = ([X_val, ART_val], Y_val),
    epochs = config_to_experiments['epochs'],
    batch_size = config_to_experiments['batch_size'],
    verbose = 1
)

Epoch 1/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 477.5303 - mae: 16.5064 - val_loss: 224.4419 - val_mae: 10.3250
Epoch 2/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 212.4445 - mae: 10.2347 - val_loss: 206.5938 - val_mae: 9.5032
Epoch 3/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 186.2999 - mae: 9.3672 - val_loss: 203.6769 - val_mae: 9.2187
Epoch 4/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 174.8446 - mae: 8.9680 - val_loss: 202.6079 - val_mae: 9.1164
Epoch 5/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 169.5852 - mae: 8.7881 - val_loss: 201.8524 - val_mae: 9.0148
Epoch 6/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 166.9588 - mae: 8.6797 - val_loss: 201.2673 - val_mae: 8.9707
Epoch 7/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 164.5143 - mae: 8.6109 - val_loss: 203.5320 - val_mae: 9.0302
Epoch 8/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 162.5556 - mae: 8.5293 - val_loss: 202.2339 - val_mae: 9.0129
Epoch 9/50
123/123 ━

Сделаем логирование по эпохам обучения

In [193]:
for epoch, (train_mae_metric, val_mae_metric) in enumerate(zip(
    learning_history.history['mae'],
    learning_history.history['val_mae']
    )):

    logger.report_scalar('MAE', 'train', value = train_mae_metric, iteration = epoch)
    logger.report_scalar('MAE', 'val', value = val_mae_metric, iteration = epoch)

In [194]:
for epoch, (train_loss, val_loss) in enumerate(zip(
    learning_history.history['loss'],
    learning_history.history['val_loss']
    )):

    logger.report_scalar('MSE loss', 'train', value = train_loss, iteration = epoch)
    logger.report_scalar('MSE loss', 'val', value = val_loss, iteration = epoch)

In [195]:
logging.info('Проверка точности модели на test')

INFO:root:Проверка точности модели на test


In [196]:
y_pred = model_task_1.predict(
    [X_test, ART_test]
).flatten()

mae_mark_test = mean_absolute_error(Y_test, y_pred)
rmse_mark_test = np.sqrt(np.mean((Y_test - y_pred)**2))

421/421 ━━━━━━━━━━━━━━━━━━━━ 1s 968us/step


In [197]:
logging.info(f'\nTEST_MAE = {mae_mark_test}\nTEST_RMSE = {rmse_mark_test}')

logger.report_single_value(name = 'TEST_MAE', value = mae_mark_test)
logger.report_single_value(name = 'TEST_RMSE', value = rmse_mark_test)

INFO:root:
TEST_MAE = 8.593467712402344
TEST_RMSE = 14.119208267121545


## Сохранение моделей

In [ ]:
NAME = "FCN_model_v" + str(6)
PATH = f'/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/DL/FCN_models_task_1/{NAME}.keras'

model_task_1.save(PATH)

In [199]:
task_1.update_output_model(
    model_path = PATH,
    model_name = NAME
)

logging.info(f'Обучение модели {NAME} завершено успешно')
task_1.close()

INFO:root:Обучение модели FCN_model_v4 завершено успешно
█████████████████████████████████ 100% | 8.27/8.27 MB [00:03<00:00,  2.67MB/s]: 
